In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from sklearn.mixture import GaussianMixture
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [25]:
class RiemannianVAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=2, hidden_dims=[512, 256]):
        super().__init__()
        self.latent_dim = latent_dim
        self.input_dim = input_dim

        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.Tanh()
            ])
            prev_dim = h_dim
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)

        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.Tanh()
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(hidden_dims[0], input_dim))
        decoder_layers.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

    def compute_metric_tensor_simple(self, z):
        """Simplified metric computation using finite differences"""
        batch_size = z.shape[0]
        M = torch.zeros(batch_size, self.latent_dim, self.latent_dim).to(z.device)

        eps = 1e-3

        with torch.no_grad():
            f0 = self.decode(z)

            for i in range(self.latent_dim):
                z_plus = z.clone()
                z_plus[:, i] += eps
                f_plus = self.decode(z_plus)
                grad_i = (f_plus - f0) / eps

                for j in range(self.latent_dim):
                    z_plus_j = z.clone()
                    z_plus_j[:, j] += eps
                    f_plus_j = self.decode(z_plus_j)
                    grad_j = (f_plus_j - f0) / eps

                    M[:, i, j] = torch.sum(grad_i * grad_j, dim=1)

        eye = torch.eye(self.latent_dim, device=z.device).unsqueeze(0)
        M = M + 1e-3 * eye

        return M

    def log_density(self, z):
        """Compute log density at point z (unnormalized posterior)"""
        with torch.no_grad():
            recon = self.decode(z.unsqueeze(0))
            # Simple prior: standard Gaussian
            log_prior = -0.5 * torch.sum(z**2)
            # Likelihood approximation based on reconstruction quality
            log_likelihood = -torch.sum((recon - 0.5)**2)  # Simplified
            return log_prior + log_likelihood

In [26]:
class AdaptiveMCMC:
    """
    Adaptive Metropolis-Hastings with Riemannian metric.
    Adapts step size to maintain target acceptance rate.
    """
    def __init__(self, model, target_accept=0.574, adapt_interval=50):
        self.model = model
        self.target_accept = target_accept  # Optimal for high-D spaces
        self.adapt_interval = adapt_interval
        self.step_size = 0.1
        self.accepts = []

    def propose(self, z_current):
        """Generate proposal using Riemannian metric"""
        M = self.model.compute_metric_tensor_simple(z_current.unsqueeze(0))

        try:
            # Use inverse metric for proposal covariance
            M_inv = torch.inverse(M[0] + 1e-4 * torch.eye(2).to(device))
            L = torch.linalg.cholesky(M_inv)
            proposal = z_current + self.step_size * (L @ torch.randn(2, 1).to(device)).squeeze()
            return proposal
        except:
            # Fallback to isotropic Gaussian
            return z_current + self.step_size * torch.randn_like(z_current)

    def accept_reject(self, z_current, z_proposal):
        """Metropolis-Hastings acceptance"""
        log_prob_current = self.model.log_density(z_current)
        log_prob_proposal = self.model.log_density(z_proposal)

        # Acceptance probability
        log_alpha = log_prob_proposal - log_prob_current

        if torch.log(torch.rand(1).to(device)) < log_alpha:
            return z_proposal, True
        else:
            return z_current, False

    def adapt_step_size(self):
        """Adapt step size based on recent acceptance rate"""
        if len(self.accepts) >= self.adapt_interval:
            recent_rate = np.mean(self.accepts[-self.adapt_interval:])

            # Robbins-Monro adaptation
            if recent_rate > self.target_accept:
                self.step_size *= 1.02  # Increase step size
            else:
                self.step_size *= 0.98  # Decrease step size

            # Keep step size in reasonable range
            self.step_size = np.clip(self.step_size, 0.01, 0.5)

    def sample(self, z_start, n_steps=100, burn_in=50):
        """Run adaptive MCMC chain"""
        samples = [z_start.clone()]

        for step in range(n_steps + burn_in):
            z_current = samples[-1]
            z_proposal = self.propose(z_current)
            z_next, accepted = self.accept_reject(z_current, z_proposal)

            samples.append(z_next)
            self.accepts.append(1 if accepted else 0)

            # Adapt step size periodically
            if (step + 1) % self.adapt_interval == 0:
                self.adapt_step_size()

        # Return samples after burn-in
        return torch.stack(samples[burn_in:])

In [27]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """VAE loss = Reconstruction + beta * KL divergence"""
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD, BCE, KLD

In [28]:
def train_vae(model, train_loader, epochs=100, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    losses = []

    for epoch in range(epochs):
        beta = min(1.0, epoch / 50)

        total_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.view(-1, 784).to(device)
            optimizer.zero_grad()

            recon, mu, logvar, z = model(data)
            loss, bce, kld = vae_loss(recon, data, mu, logvar, beta)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader.dataset)
        losses.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Beta: {beta:.3f}')

    return losses

In [29]:
def compute_geodesic(model, z0, z1, n_waypoints=50, n_steps=200, lr=0.03):
    """Compute geodesic path between z0 and z1"""
    model.eval()

    alphas = torch.linspace(0, 1, n_waypoints).to(device)
    waypoints_init = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)

    interior_waypoints = nn.Parameter(waypoints_init[1:-1].clone())
    optimizer = optim.Adam([interior_waypoints], lr=lr)

    for step in range(n_steps):
        optimizer.zero_grad()
        waypoints = torch.cat([z0.unsqueeze(0), interior_waypoints, z1.unsqueeze(0)], dim=0)

        energy = torch.tensor(0.0, device=device, requires_grad=True)

        for i in range(n_waypoints - 1):
            dz = waypoints[i+1] - waypoints[i]
            z_mid = (waypoints[i] + waypoints[i+1]) / 2
            M = model.compute_metric_tensor_simple(z_mid.unsqueeze(0))
            dist_sq = torch.sum(dz * (M[0] @ dz))
            energy = energy + torch.sqrt(dist_sq + 1e-8)

        energy.backward()
        optimizer.step()

    with torch.no_grad():
        final_waypoints = torch.cat([z0.unsqueeze(0), interior_waypoints, z1.unsqueeze(0)], dim=0)

    return final_waypoints

In [30]:
def plot_latent_space(model, test_loader, save_path='figure_latent_space.png'):
    """Plot 2D latent space with class colors"""
    model.eval()

    latents = []
    labels = []

    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels.append(label)

    latents = torch.cat(latents, dim=0).numpy()
    labels = torch.cat(labels, dim=0).numpy()

    plt.figure(figsize=(8, 8))
    colors = {0: 'blue', 1: 'red', 8: 'green'}
    for digit in [0, 1, 8]:
        mask = labels == digit
        plt.scatter(latents[mask, 0], latents[mask, 1],
                   label=f'Digit {digit}', alpha=0.6, s=20, c=colors[digit])
    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Latent Space Representation', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [31]:
def plot_variance_landscape(model, test_loader, save_path='figure_variance_analysis.png'):
    """Plot metric determinant landscape"""
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    grid_size = 60
    x_min, x_max = latents[:, 0].min() - 1, latents[:, 0].max() + 1
    y_min, y_max = latents[:, 1].min() - 1, latents[:, 1].max() + 1
    x = np.linspace(x_min, x_max, grid_size)
    y = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x, y)
    Z_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

    print("  Computing metric landscape...")
    metric_dets = []

    z_tensor = torch.FloatTensor(Z_grid).to(device)
    batch_size = 100

    for i in range(0, len(z_tensor), batch_size):
        batch = z_tensor[i:i+batch_size]
        M = model.compute_metric_tensor_simple(batch)
        det = torch.det(M)
        metric_dets.append(det.cpu().numpy())

    metric_dets = np.concatenate(metric_dets)
    metric_dets = metric_dets.reshape(grid_size, grid_size)

    plt.figure(figsize=(10, 8))
    plt.contourf(X, Y, -np.log(metric_dets + 1e-6), levels=20, cmap='gray')
    plt.colorbar(label='-log det(M) [darker = data-rich]')
    plt.scatter(latents[:, 0], latents[:, 1], c='red', s=1, alpha=0.3)

    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Metric Landscape', fontsize=14)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [32]:
def plot_interpolation(model, z0, z1, save_path='figure_interpolation.png'):
    """Compare Euclidean vs Geodesic interpolation"""
    model.eval()

    n_frames = 10
    alphas = torch.linspace(0, 1, n_frames).to(device)

    z_euclidean = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)

    print("   Computing geodesic...")
    geodesic = compute_geodesic(model, z0, z1, n_waypoints=50, n_steps=200)

    indices = np.linspace(0, len(geodesic)-1, n_frames).astype(int)
    z_geodesic = geodesic[indices]

    with torch.no_grad():
        imgs_euclidean = model.decode(z_euclidean).cpu().view(n_frames, 28, 28)
        imgs_geodesic = model.decode(z_geodesic).cpu().view(n_frames, 28, 28)

    fig, axes = plt.subplots(2, n_frames, figsize=(15, 3))

    for i in range(n_frames):
        axes[0, i].imshow(imgs_euclidean[i], cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].text(-5, 14, 'Euclidean', fontsize=11, rotation=90, va='center')

        axes[1, i].imshow(imgs_geodesic[i], cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].text(-5, 14, 'Geodesic', fontsize=11, rotation=90, va='center')

    plt.suptitle('Interpolation: Euclidean (top) vs Geodesic (bottom)', fontsize=12, y=0.98)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [33]:
def plot_random_walk_comparison(model, z_start, n_steps=100,
                                save_path='figure_random_walk.png'):
    """Compare Euclidean, naive Riemannian, and Adaptive MCMC"""
    model.eval()

    print("  Computing random walks...")

    step_size = 0.15

    # 1. Euclidean random walk
    z_euclidean = [z_start.clone()]
    for _ in range(n_steps):
        step = torch.randn_like(z_start) * step_size
        z_euclidean.append(z_euclidean[-1] + step)

    # 2. Naive Riemannian random walk
    z_riemannian = [z_start.clone()]
    for _ in range(n_steps):
        z_current = z_riemannian[-1].unsqueeze(0)
        M = model.compute_metric_tensor_simple(z_current)
        M_inv = torch.inverse(M[0] + 1e-4 * torch.eye(2).to(device))

        try:
            L = torch.linalg.cholesky(M_inv)
            step = (L @ torch.randn(2, 1).to(device)).squeeze() * step_size
            z_riemannian.append(z_riemannian[-1] + step)
        except:
            step = torch.randn_like(z_start) * step_size
            z_riemannian.append(z_riemannian[-1] + step)

    # 3. Adaptive MCMC
    print("  Running Adaptive MCMC...")
    mcmc = AdaptiveMCMC(model, target_accept=0.574, adapt_interval=20)
    z_mcmc_samples = mcmc.sample(z_start, n_steps=n_steps, burn_in=20)

    # Convert to numpy
    z_euclidean = torch.stack(z_euclidean).cpu().numpy()
    z_riemannian = torch.stack(z_riemannian).cpu().numpy()
    z_mcmc = z_mcmc_samples.cpu().numpy()

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Euclidean
    axes[0].plot(z_euclidean[:, 0], z_euclidean[:, 1], 'r-', alpha=0.6, linewidth=1.5)
    axes[0].scatter(z_euclidean[0, 0], z_euclidean[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[0].scatter(z_euclidean[-1, 0], z_euclidean[-1, 1], c='red', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[0].set_xlabel('z₁', fontsize=12)
    axes[0].set_ylabel('z₂', fontsize=12)
    axes[0].set_title('Euclidean Walk\n(escapes manifold)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Naive Riemannian
    axes[1].plot(z_riemannian[:, 0], z_riemannian[:, 1], 'b-', alpha=0.6, linewidth=1.5)
    axes[1].scatter(z_riemannian[0, 0], z_riemannian[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[1].scatter(z_riemannian[-1, 0], z_riemannian[-1, 1], c='blue', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[1].set_xlabel('z₁', fontsize=12)
    axes[1].set_ylabel('z₂', fontsize=12)
    axes[1].set_title('Naive Riemannian Walk\n(metric-aware)', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Adaptive MCMC
    axes[2].plot(z_mcmc[:, 0], z_mcmc[:, 1], 'purple', alpha=0.6, linewidth=1.5)
    axes[2].scatter(z_mcmc[0, 0], z_mcmc[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[2].scatter(z_mcmc[-1, 0], z_mcmc[-1, 1], c='purple', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[2].set_xlabel('z₁', fontsize=12)
    axes[2].set_ylabel('z₂', fontsize=12)
    axes[2].set_title(f'Adaptive MCMC\n(accept rate: {np.mean(mcmc.accepts):.2%})', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")
    print(f"  Final MCMC step size: {mcmc.step_size:.4f}")
    print(f"  Overall acceptance rate: {np.mean(mcmc.accepts):.2%}")

In [34]:
def plot_mcmc_diagnostics(model, z_start, n_steps=500, save_path='figure_mcmc_diagnostics.png'):
    """Plot MCMC diagnostics: trace, acceptance rate, step size adaptation"""
    model.eval()

    print("  Running MCMC for diagnostics...")
    mcmc = AdaptiveMCMC(model, target_accept=0.574, adapt_interval=20)
    samples = mcmc.sample(z_start, n_steps=n_steps, burn_in=100)

    samples_np = samples.cpu().numpy()
    accepts = np.array(mcmc.accepts)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Trace plot
    axes[0, 0].plot(samples_np[:, 0], alpha=0.7, label='z₁')
    axes[0, 0].plot(samples_np[:, 1], alpha=0.7, label='z₂')
    axes[0, 0].set_xlabel('Iteration', fontsize=11)
    axes[0, 0].set_ylabel('Value', fontsize=11)
    axes[0, 0].set_title('MCMC Trace Plot', fontsize=12)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Acceptance rate over time
    window = 50
    accept_rate = np.convolve(accepts, np.ones(window)/window, mode='valid')
    axes[0, 1].plot(accept_rate, color='green')
    axes[0, 1].axhline(y=mcmc.target_accept, color='red', linestyle='--',
                       label=f'Target: {mcmc.target_accept:.1%}')
    axes[0, 1].set_xlabel('Iteration', fontsize=11)
    axes[0, 1].set_ylabel('Acceptance Rate', fontsize=11)
    axes[0, 1].set_title(f'Acceptance Rate (window={window})', fontsize=12)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 1])

    # 2D trajectory
    axes[1, 0].plot(samples_np[:, 0], samples_np[:, 1], 'b-', alpha=0.3, linewidth=0.5)
    axes[1, 0].scatter(samples_np[::10, 0], samples_np[::10, 1],
                      c=range(0, len(samples_np), 10), cmap='viridis', s=10, alpha=0.6)
    axes[1, 0].scatter(samples_np[0, 0], samples_np[0, 1], c='green',
                      s=200, marker='o', edgecolors='black', linewidths=2, zorder=5)
    axes[1, 0].set_xlabel('z₁', fontsize=11)
    axes[1, 0].set_ylabel('z₂', fontsize=11)
    axes[1, 0].set_title('MCMC Trajectory in Latent Space', fontsize=12)
    axes[1, 0].grid(True, alpha=0.3)

    # Histogram of samples
    axes[1, 1].hist2d(samples_np[:, 0], samples_np[:, 1], bins=30, cmap='Blues')
    axes[1, 1].set_xlabel('z₁', fontsize=11)
    axes[1, 1].set_ylabel('z₂', fontsize=11)
    axes[1, 1].set_title('Sample Density (2D Histogram)', fontsize=12)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [35]:
def plot_rbf_centers(model, train_loader, K=15, save_path='figure_rbf_centers.png'):
    """Plot latent space with RBF centers"""
    model.eval()

    latents = []
    labels = []

    with torch.no_grad():
        for data, label in train_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels.append(label)

    Z = torch.cat(latents, dim=0).numpy()
    Y = torch.cat(labels, dim=0).numpy()

    gmm = GaussianMixture(n_components=K, covariance_type='spherical', random_state=0)
    gmm.fit(Z)

    centers = gmm.means_
    lambda_rbf = np.sqrt(gmm.covariances_).mean()

    plt.figure(figsize=(8, 8))
    plt.scatter(Z[:, 0], Z[:, 1], c=Y, cmap='tab10', s=8, alpha=0.35)
    plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='x',
                s=150, linewidths=2, label='RBF Centers')

    for c in centers:
        circle = Circle(c, radius=lambda_rbf, fill=False, linestyle='--',
                       edgecolor='red', alpha=0.6)
        plt.gca().add_patch(circle)

    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Latent Space Coverage via RBF Variance Network', fontsize=14)
    plt.axis('equal')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path}")

In [36]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in [0, 1, 8]]
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in [0, 1, 8]]

train_dataset = Subset(train_dataset, train_indices)
test_dataset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [37]:
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

Train samples: 18516
Test samples:  3089


In [38]:
model = RiemannianVAE(
    input_dim=784,
    latent_dim=2,
    hidden_dims=[512, 256]
).to(device)

In [39]:
losses = train_vae(
    model,
    train_loader,
    epochs=100,
    lr=1e-3
)

Epoch 10/100, Loss: 114.4512, Beta: 0.180
Epoch 20/100, Loss: 112.9639, Beta: 0.380
Epoch 30/100, Loss: 112.8579, Beta: 0.580
Epoch 40/100, Loss: 113.3712, Beta: 0.780
Epoch 50/100, Loss: 113.6857, Beta: 0.980
Epoch 60/100, Loss: 112.7168, Beta: 1.000
Epoch 70/100, Loss: 111.8373, Beta: 1.000
Epoch 80/100, Loss: 110.9995, Beta: 1.000
Epoch 90/100, Loss: 110.5439, Beta: 1.000
Epoch 100/100, Loss: 109.9685, Beta: 1.000


In [40]:
torch.save(model.state_dict(), "riemannian_vae.pth")

In [41]:
plot_latent_space(
    model,
    test_loader,
    save_path="figure_latent_space.png"
)

Saved: figure_latent_space.png


In [42]:
plot_variance_landscape(
    model,
    test_loader,
    save_path="figure_variance_analysis.png"
)

  Computing metric landscape...
Saved: figure_variance_analysis.png


In [43]:
model.eval()
with torch.no_grad():
    data_iter = iter(test_loader)
    data, labels = next(data_iter)
    data = data.view(-1, 784).to(device)
    mu, _ = model.encode(data)

    # pick two different digits
    idx0 = (labels == 0).nonzero()[0].item()
    idx1 = (labels == 8).nonzero()[0].item()

    z0 = mu[idx0]
    z1 = mu[idx1]

In [44]:
plot_interpolation(
    model,
    z0,
    z1,
    save_path="figure_interpolation.png"
)

   Computing geodesic...
 Saved: figure_interpolation.png


In [45]:
plot_random_walk_comparison(
    model,
    z_start=z0,
    n_steps=150,
    save_path="figure_random_walk.png"
)

  Computing random walks...
  Running Adaptive MCMC...
 Saved: figure_random_walk.png
  Final MCMC step size: 0.1172
  Overall acceptance rate: 98.24%


In [46]:
plot_mcmc_diagnostics(
    model,
    z_start=z0,
    n_steps=600,
    save_path="figure_mcmc_diagnostics.png"
)

  Running MCMC for diagnostics...
Saved: figure_mcmc_diagnostics.png


In [47]:
plot_rbf_centers(
    model,
    train_loader,
    K=15,
    save_path="figure_rbf_centers.png"
)

✓ Saved: figure_rbf_centers.png


In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from sklearn.mixture import GaussianMixture
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [49]:
class RiemannianVAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=2, hidden_dims=[512, 256]):
        super().__init__()
        self.latent_dim = latent_dim
        self.input_dim = input_dim

        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.Tanh()
            ])
            prev_dim = h_dim
        self.encoder = nn.Sequential(*encoder_layers)
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)

        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.Tanh()
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(hidden_dims[0], input_dim))
        decoder_layers.append(nn.Sigmoid())
        self.decoder = nn.Sequential(*decoder_layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

    def compute_metric_tensor_simple(self, z):
        """Simplified metric computation using finite differences"""
        batch_size = z.shape[0]
        M = torch.zeros(batch_size, self.latent_dim, self.latent_dim).to(z.device)

        eps = 1e-3

        with torch.no_grad():
            f0 = self.decode(z)

            for i in range(self.latent_dim):
                z_plus = z.clone()
                z_plus[:, i] += eps
                f_plus = self.decode(z_plus)
                grad_i = (f_plus - f0) / eps

                for j in range(self.latent_dim):
                    z_plus_j = z.clone()
                    z_plus_j[:, j] += eps
                    f_plus_j = self.decode(z_plus_j)
                    grad_j = (f_plus_j - f0) / eps

                    M[:, i, j] = torch.sum(grad_i * grad_j, dim=1)

        eye = torch.eye(self.latent_dim, device=z.device).unsqueeze(0)
        M = M + 1e-3 * eye

        return M

    def log_density(self, z):
        """Compute log density at point z (unnormalized posterior)"""
        with torch.no_grad():
            recon = self.decode(z.unsqueeze(0))
            # Simple prior: standard Gaussian
            log_prior = -0.5 * torch.sum(z**2)
            # Likelihood approximation based on reconstruction quality
            log_likelihood = -torch.sum((recon - 0.5)**2)  # Simplified
            return log_prior + log_likelihood

In [50]:
class AdaptiveMCMC:
    """
    Adaptive Metropolis-Hastings with Riemannian metric.
    Adapts step size to maintain target acceptance rate.
    """
    def __init__(self, model, target_accept=0.574, adapt_interval=50):
        self.model = model
        self.target_accept = target_accept  # Optimal for high-D spaces
        self.adapt_interval = adapt_interval
        self.step_size = 0.1
        self.accepts = []

    def propose(self, z_current):
        """Generate proposal using Riemannian metric"""
        M = self.model.compute_metric_tensor_simple(z_current.unsqueeze(0))

        try:
            # Use inverse metric for proposal covariance
            M_inv = torch.inverse(M[0] + 1e-4 * torch.eye(2).to(device))
            L = torch.linalg.cholesky(M_inv)
            proposal = z_current + self.step_size * (L @ torch.randn(2, 1).to(device)).squeeze()
            return proposal
        except:
            # Fallback to isotropic Gaussian
            return z_current + self.step_size * torch.randn_like(z_current)

    def accept_reject(self, z_current, z_proposal):
        """Metropolis-Hastings acceptance"""
        log_prob_current = self.model.log_density(z_current)
        log_prob_proposal = self.model.log_density(z_proposal)

        # Acceptance probability
        log_alpha = log_prob_proposal - log_prob_current

        if torch.log(torch.rand(1).to(device)) < log_alpha:
            return z_proposal, True
        else:
            return z_current, False

    def adapt_step_size(self):
        """Adapt step size based on recent acceptance rate"""
        if len(self.accepts) >= self.adapt_interval:
            recent_rate = np.mean(self.accepts[-self.adapt_interval:])

            # Robbins-Monro adaptation
            if recent_rate > self.target_accept:
                self.step_size *= 1.02  # Increase step size
            else:
                self.step_size *= 0.98  # Decrease step size

            # Keep step size in reasonable range
            self.step_size = np.clip(self.step_size, 0.01, 0.5)

    def sample(self, z_start, n_steps=100, burn_in=50):
        """Run adaptive MCMC chain"""
        samples = [z_start.clone()]

        for step in range(n_steps + burn_in):
            z_current = samples[-1]
            z_proposal = self.propose(z_current)
            z_next, accepted = self.accept_reject(z_current, z_proposal)

            samples.append(z_next)
            self.accepts.append(1 if accepted else 0)

            # Adapt step size periodically
            if (step + 1) % self.adapt_interval == 0:
                self.adapt_step_size()

        # Return samples after burn-in
        return torch.stack(samples[burn_in:])

In [51]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    """VAE loss = Reconstruction + beta * KL divergence"""
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD, BCE, KLD

In [52]:

def train_vae(model, train_loader, epochs=100, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    losses = []

    for epoch in range(epochs):
        beta = min(1.0, epoch / 50)

        total_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.view(-1, 784).to(device)
            optimizer.zero_grad()

            recon, mu, logvar, z = model(data)
            loss, bce, kld = vae_loss(recon, data, mu, logvar, beta)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader.dataset)
        losses.append(avg_loss)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Beta: {beta:.3f}')

    return losses

In [53]:
def compute_geodesic(model, z0, z1, n_waypoints=50, n_steps=200, lr=0.03):
    """Compute geodesic path between z0 and z1"""
    model.eval()

    alphas = torch.linspace(0, 1, n_waypoints).to(device)
    waypoints_init = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)

    interior_waypoints = nn.Parameter(waypoints_init[1:-1].clone())
    optimizer = optim.Adam([interior_waypoints], lr=lr)

    for step in range(n_steps):
        optimizer.zero_grad()
        waypoints = torch.cat([z0.unsqueeze(0), interior_waypoints, z1.unsqueeze(0)], dim=0)

        energy = torch.tensor(0.0, device=device, requires_grad=True)

        for i in range(n_waypoints - 1):
            dz = waypoints[i+1] - waypoints[i]
            z_mid = (waypoints[i] + waypoints[i+1]) / 2
            M = model.compute_metric_tensor_simple(z_mid.unsqueeze(0))
            dist_sq = torch.sum(dz * (M[0] @ dz))
            energy = energy + torch.sqrt(dist_sq + 1e-8)

        energy.backward()
        optimizer.step()

    with torch.no_grad():
        final_waypoints = torch.cat([z0.unsqueeze(0), interior_waypoints, z1.unsqueeze(0)], dim=0)

    return final_waypoints

In [54]:
def plot_latent_space(model, test_loader, save_path='figure_latent_space.png'):
    """Plot 2D latent space with class colors"""
    model.eval()

    latents = []
    labels = []

    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels.append(label)

    latents = torch.cat(latents, dim=0).numpy()
    labels = torch.cat(labels, dim=0).numpy()

    plt.figure(figsize=(8, 8))
    colors = {0: 'blue', 1: 'red', 8: 'green'}
    for digit in [0, 1, 8]:
        mask = labels == digit
        plt.scatter(latents[mask, 0], latents[mask, 1],
                   label=f'Digit {digit}', alpha=0.6, s=20, c=colors[digit])
    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Latent Space Representation', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [55]:
def plot_variance_landscape(model, test_loader, save_path='figure_variance_analysis.png'):
    """Plot metric determinant landscape"""
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    grid_size = 60
    x_min, x_max = latents[:, 0].min() - 1, latents[:, 0].max() + 1
    y_min, y_max = latents[:, 1].min() - 1, latents[:, 1].max() + 1
    x = np.linspace(x_min, x_max, grid_size)
    y = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x, y)
    Z_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

    print("  Computing metric landscape...")
    metric_dets = []

    z_tensor = torch.FloatTensor(Z_grid).to(device)
    batch_size = 100

    for i in range(0, len(z_tensor), batch_size):
        batch = z_tensor[i:i+batch_size]
        M = model.compute_metric_tensor_simple(batch)
        det = torch.det(M)
        metric_dets.append(det.cpu().numpy())

    metric_dets = np.concatenate(metric_dets)
    metric_dets = metric_dets.reshape(grid_size, grid_size)

    plt.figure(figsize=(10, 8))
    plt.contourf(X, Y, -np.log(metric_dets + 1e-6), levels=20, cmap='gray')
    plt.colorbar(label='-log det(M) [darker = data-rich]')
    plt.scatter(latents[:, 0], latents[:, 1], c='red', s=1, alpha=0.3)

    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Metric Landscape', fontsize=14)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [56]:
def plot_interpolation(model, z0, z1, save_path='figure_interpolation.png'):
    """Compare Euclidean vs Geodesic interpolation"""
    model.eval()

    n_frames = 10
    alphas = torch.linspace(0, 1, n_frames).to(device)

    z_euclidean = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)

    print("   Computing geodesic...")
    geodesic = compute_geodesic(model, z0, z1, n_waypoints=50, n_steps=200)

    indices = np.linspace(0, len(geodesic)-1, n_frames).astype(int)
    z_geodesic = geodesic[indices]

    with torch.no_grad():
        imgs_euclidean = model.decode(z_euclidean).cpu().view(n_frames, 28, 28)
        imgs_geodesic = model.decode(z_geodesic).cpu().view(n_frames, 28, 28)

    fig, axes = plt.subplots(2, n_frames, figsize=(15, 3))

    for i in range(n_frames):
        axes[0, i].imshow(imgs_euclidean[i], cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].text(-5, 14, 'Euclidean', fontsize=11, rotation=90, va='center')

        axes[1, i].imshow(imgs_geodesic[i], cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].text(-5, 14, 'Geodesic', fontsize=11, rotation=90, va='center')

    plt.suptitle('Interpolation: Euclidean (top) vs Geodesic (bottom)', fontsize=12, y=0.98)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [57]:
def plot_random_walk_comparison(model, z_start, n_steps=100,
                                save_path='figure_random_walk.png'):
    """Compare Euclidean, naive Riemannian, and Adaptive MCMC"""
    model.eval()

    print("  Computing random walks...")

    step_size = 0.15

    # 1. Euclidean random walk
    z_euclidean = [z_start.clone()]
    for _ in range(n_steps):
        step = torch.randn_like(z_start) * step_size
        z_euclidean.append(z_euclidean[-1] + step)

    # 2. Naive Riemannian random walk
    z_riemannian = [z_start.clone()]
    for _ in range(n_steps):
        z_current = z_riemannian[-1].unsqueeze(0)
        M = model.compute_metric_tensor_simple(z_current)
        M_inv = torch.inverse(M[0] + 1e-4 * torch.eye(2).to(device))

        try:
            L = torch.linalg.cholesky(M_inv)
            step = (L @ torch.randn(2, 1).to(device)).squeeze() * step_size
            z_riemannian.append(z_riemannian[-1] + step)
        except:
            step = torch.randn_like(z_start) * step_size
            z_riemannian.append(z_riemannian[-1] + step)

    # 3. Adaptive MCMC
    print("  Running Adaptive MCMC...")
    mcmc = AdaptiveMCMC(model, target_accept=0.574, adapt_interval=20)
    z_mcmc_samples = mcmc.sample(z_start, n_steps=n_steps, burn_in=20)

    # Convert to numpy
    z_euclidean = torch.stack(z_euclidean).cpu().numpy()
    z_riemannian = torch.stack(z_riemannian).cpu().numpy()
    z_mcmc = z_mcmc_samples.cpu().numpy()

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Euclidean
    axes[0].plot(z_euclidean[:, 0], z_euclidean[:, 1], 'r-', alpha=0.6, linewidth=1.5)
    axes[0].scatter(z_euclidean[0, 0], z_euclidean[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[0].scatter(z_euclidean[-1, 0], z_euclidean[-1, 1], c='red', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[0].set_xlabel('z₁', fontsize=12)
    axes[0].set_ylabel('z₂', fontsize=12)
    axes[0].set_title('Euclidean Walk\n(escapes manifold)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Naive Riemannian
    axes[1].plot(z_riemannian[:, 0], z_riemannian[:, 1], 'b-', alpha=0.6, linewidth=1.5)
    axes[1].scatter(z_riemannian[0, 0], z_riemannian[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[1].scatter(z_riemannian[-1, 0], z_riemannian[-1, 1], c='blue', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[1].set_xlabel('z₁', fontsize=12)
    axes[1].set_ylabel('z₂', fontsize=12)
    axes[1].set_title('Naive Riemannian Walk\n(metric-aware)', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Adaptive MCMC
    axes[2].plot(z_mcmc[:, 0], z_mcmc[:, 1], 'purple', alpha=0.6, linewidth=1.5)
    axes[2].scatter(z_mcmc[0, 0], z_mcmc[0, 1], c='green', s=150,
                   marker='o', label='Start', edgecolors='black', linewidths=2, zorder=5)
    axes[2].scatter(z_mcmc[-1, 0], z_mcmc[-1, 1], c='purple', s=150,
                   marker='X', label='End', edgecolors='black', linewidths=2, zorder=5)
    axes[2].set_xlabel('z₁', fontsize=12)
    axes[2].set_ylabel('z₂', fontsize=12)
    axes[2].set_title(f'Adaptive MCMC\n(accept rate: {np.mean(mcmc.accepts):.2%})', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")
    print(f"  Final MCMC step size: {mcmc.step_size:.4f}")
    print(f"  Overall acceptance rate: {np.mean(mcmc.accepts):.2%}")

In [58]:
def plot_mcmc_diagnostics(model, z_start, n_steps=500, save_path='figure_mcmc_diagnostics.png'):
    """Plot MCMC diagnostics: trace, acceptance rate, step size adaptation"""
    model.eval()

    print("  Running MCMC for diagnostics...")
    mcmc = AdaptiveMCMC(model, target_accept=0.574, adapt_interval=20)
    samples = mcmc.sample(z_start, n_steps=n_steps, burn_in=100)

    samples_np = samples.cpu().numpy()
    accepts = np.array(mcmc.accepts)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Trace plot
    axes[0, 0].plot(samples_np[:, 0], alpha=0.7, label='z₁')
    axes[0, 0].plot(samples_np[:, 1], alpha=0.7, label='z₂')
    axes[0, 0].set_xlabel('Iteration', fontsize=11)
    axes[0, 0].set_ylabel('Value', fontsize=11)
    axes[0, 0].set_title('MCMC Trace Plot', fontsize=12)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Acceptance rate over time
    window = 50
    accept_rate = np.convolve(accepts, np.ones(window)/window, mode='valid')
    axes[0, 1].plot(accept_rate, color='green')
    axes[0, 1].axhline(y=mcmc.target_accept, color='red', linestyle='--',
                       label=f'Target: {mcmc.target_accept:.1%}')
    axes[0, 1].set_xlabel('Iteration', fontsize=11)
    axes[0, 1].set_ylabel('Acceptance Rate', fontsize=11)
    axes[0, 1].set_title(f'Acceptance Rate (window={window})', fontsize=12)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 1])

    # 2D trajectory
    axes[1, 0].plot(samples_np[:, 0], samples_np[:, 1], 'b-', alpha=0.3, linewidth=0.5)
    axes[1, 0].scatter(samples_np[::10, 0], samples_np[::10, 1],
                      c=range(0, len(samples_np), 10), cmap='viridis', s=10, alpha=0.6)
    axes[1, 0].scatter(samples_np[0, 0], samples_np[0, 1], c='green',
                      s=200, marker='o', edgecolors='black', linewidths=2, zorder=5)
    axes[1, 0].set_xlabel('z₁', fontsize=11)
    axes[1, 0].set_ylabel('z₂', fontsize=11)
    axes[1, 0].set_title('MCMC Trajectory in Latent Space', fontsize=12)
    axes[1, 0].grid(True, alpha=0.3)

    # Histogram of samples
    axes[1, 1].hist2d(samples_np[:, 0], samples_np[:, 1], bins=30, cmap='Blues')
    axes[1, 1].set_xlabel('z₁', fontsize=11)
    axes[1, 1].set_ylabel('z₂', fontsize=11)
    axes[1, 1].set_title('Sample Density (2D Histogram)', fontsize=12)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [59]:
def plot_rbf_centers(model, train_loader, K=15, save_path='figure_rbf_centers.png'):
    """Plot latent space with RBF centers"""
    model.eval()

    latents = []
    labels = []

    with torch.no_grad():
        for data, label in train_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
            labels.append(label)

    Z = torch.cat(latents, dim=0).numpy()
    Y = torch.cat(labels, dim=0).numpy()

    gmm = GaussianMixture(n_components=K, covariance_type='spherical', random_state=0)
    gmm.fit(Z)

    centers = gmm.means_
    lambda_rbf = np.sqrt(gmm.covariances_).mean()

    plt.figure(figsize=(8, 8))
    plt.scatter(Z[:, 0], Z[:, 1], c=Y, cmap='tab10', s=8, alpha=0.35)
    plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='x',
                s=150, linewidths=2, label='RBF Centers')

    for c in centers:
        circle = Circle(c, radius=lambda_rbf, fill=False, linestyle='--',
                       edgecolor='red', alpha=0.6)
        plt.gca().add_patch(circle)

    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Latent Space Coverage via RBF Variance Network', fontsize=14)
    plt.axis('equal')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [60]:
def plot_metric_ellipses(model, test_loader, n_points=20, save_path='figure_metric_ellipses.png'):
    """
    Visualize metric tensor as ellipses at different latent space locations.
    Ellipse shape shows local geometry: circular = isotropic, elongated = anisotropic
    """
    model.eval()

    # Get latent space bounds
    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    # Sample grid points
    x_min, x_max = latents[:, 0].min(), latents[:, 0].max()
    y_min, y_max = latents[:, 1].min(), latents[:, 1].max()

    x_points = np.linspace(x_min, x_max, int(np.sqrt(n_points)))
    y_points = np.linspace(y_min, y_max, int(np.sqrt(n_points)))

    plt.figure(figsize=(10, 10))
    plt.scatter(latents[:, 0], latents[:, 1], c='lightgray', s=1, alpha=0.3)

    print("  Computing metric ellipses...")
    for x in x_points:
        for y in y_points:
            z = torch.FloatTensor([[x, y]]).to(device)
            M = model.compute_metric_tensor_simple(z)[0].cpu().numpy()

            # Compute eigendecomposition for ellipse
            eigvals, eigvecs = np.linalg.eigh(M)

            # Ellipse parameters
            angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
            width = 2 * np.sqrt(eigvals[0]) * 0.2  # Scale for visualization
            height = 2 * np.sqrt(eigvals[1]) * 0.2

            from matplotlib.patches import Ellipse
            ellipse = Ellipse((x, y), width, height, angle=angle,
                            fill=False, edgecolor='blue', linewidth=1.5, alpha=0.6)
            plt.gca().add_patch(ellipse)

    plt.xlabel('z₁', fontsize=12)
    plt.ylabel('z₂', fontsize=12)
    plt.title('Metric Tensor Field (Ellipses show local geometry)', fontsize=14)
    plt.axis('equal')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [61]:
def plot_curvature_scalar(model, test_loader, save_path='figure_curvature_scalar.png'):
    """
    Visualize scalar curvature (trace of metric tensor).
    High curvature = complex local geometry, low = flat
    """
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    # Create grid
    grid_size = 80
    x_min, x_max = latents[:, 0].min() - 1, latents[:, 0].max() + 1
    y_min, y_max = latents[:, 1].min() - 1, latents[:, 1].max() + 1
    x = np.linspace(x_min, x_max, grid_size)
    y = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x, y)
    Z_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

    print("  Computing scalar curvature...")
    curvatures = []

    z_tensor = torch.FloatTensor(Z_grid).to(device)
    batch_size = 100

    for i in range(0, len(z_tensor), batch_size):
        batch = z_tensor[i:i+batch_size]
        M = model.compute_metric_tensor_simple(batch)
        # Scalar curvature approximation: trace(M)
        trace = torch.einsum('bii->b', M)
        curvatures.append(trace.cpu().numpy())

    curvatures = np.concatenate(curvatures).reshape(grid_size, grid_size)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Curvature heatmap
    im1 = axes[0].contourf(X, Y, curvatures, levels=25, cmap='coolwarm')
    axes[0].scatter(latents[:, 0], latents[:, 1], c='black', s=0.5, alpha=0.3)
    axes[0].set_xlabel('z₁', fontsize=12)
    axes[0].set_ylabel('z₂', fontsize=12)
    axes[0].set_title('Scalar Curvature (Trace of Metric)', fontsize=13)
    plt.colorbar(im1, ax=axes[0], label='Tr(M)')

    # Curvature with contour lines
    im2 = axes[1].contourf(X, Y, curvatures, levels=25, cmap='viridis', alpha=0.8)
    axes[1].contour(X, Y, curvatures, levels=10, colors='white', linewidths=0.5, alpha=0.4)
    axes[1].scatter(latents[:, 0], latents[:, 1], c='red', s=1, alpha=0.4)
    axes[1].set_xlabel('z₁', fontsize=12)
    axes[1].set_ylabel('z₂', fontsize=12)
    axes[1].set_title('Curvature with Data Overlay', fontsize=13)
    plt.colorbar(im2, ax=axes[1], label='Tr(M)')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [62]:
def plot_distance_comparison(model, test_loader, save_path='figure_distance_comparison.png'):
    """
    Compare Euclidean vs Riemannian distances between pairs of points.
    Shows how geometry affects distance measurements.
    """
    model.eval()

    # Get test points
    with torch.no_grad():
        for data, labels in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)

            # Get points from different classes
            idx_0 = (labels == 0).nonzero(as_tuple=True)[0]
            idx_1 = (labels == 1).nonzero(as_tuple=True)[0]
            idx_8 = (labels == 8).nonzero(as_tuple=True)[0]

            if len(idx_0) > 10 and len(idx_1) > 10 and len(idx_8) > 10:
                z_points = torch.cat([
                    mu[idx_0[:5]],
                    mu[idx_1[:5]],
                    mu[idx_8[:5]]
                ]).cpu()
                break

    n_points = len(z_points)
    euclidean_dists = np.zeros((n_points, n_points))
    riemannian_dists = np.zeros((n_points, n_points))

    print("  Computing pairwise distances...")
    for i in range(n_points):
        for j in range(i+1, n_points):
            # Euclidean distance
            euclidean_dists[i, j] = torch.norm(z_points[i] - z_points[j]).item()

            # Riemannian distance (approximate via metric at midpoint)
            z_mid = ((z_points[i] + z_points[j]) / 2).to(device).unsqueeze(0)
            M = model.compute_metric_tensor_simple(z_mid)[0]
            dz = (z_points[j] - z_points[i]).to(device)
            riem_dist = torch.sqrt(torch.sum(dz * (M @ dz))).item()
            riemannian_dists[i, j] = riem_dist

            # Make symmetric
            euclidean_dists[j, i] = euclidean_dists[i, j]
            riemannian_dists[j, i] = riemannian_dists[i, j]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Euclidean distances
    im1 = axes[0].imshow(euclidean_dists, cmap='YlOrRd', aspect='auto')
    axes[0].set_title('Euclidean Distances', fontsize=13)
    axes[0].set_xlabel('Point Index', fontsize=11)
    axes[0].set_ylabel('Point Index', fontsize=11)
    plt.colorbar(im1, ax=axes[0])

    # Riemannian distances
    im2 = axes[1].imshow(riemannian_dists, cmap='YlOrRd', aspect='auto')
    axes[1].set_title('Riemannian Distances', fontsize=13)
    axes[1].set_xlabel('Point Index', fontsize=11)
    axes[1].set_ylabel('Point Index', fontsize=11)
    plt.colorbar(im2, ax=axes[1])

    # Ratio (Riemannian / Euclidean)
    ratio = riemannian_dists / (euclidean_dists + 1e-8)
    im3 = axes[2].imshow(ratio, cmap='RdBu_r', aspect='auto', vmin=0.5, vmax=2.0)
    axes[2].set_title('Distance Ratio (Riemannian/Euclidean)', fontsize=13)
    axes[2].set_xlabel('Point Index', fontsize=11)
    axes[2].set_ylabel('Point Index', fontsize=11)
    plt.colorbar(im3, ax=axes[2], label='Ratio')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [63]:
def plot_geodesic_vs_euclidean_paths(model, test_loader, n_pairs=5,
                                     save_path='figure_geodesic_paths.png'):
    """
    Visualize multiple geodesic vs Euclidean paths in latent space.
    Shows how paths curve around regions of high curvature.
    """
    model.eval()

    # Get diverse pairs of points
    with torch.no_grad():
        for data, labels in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)

            pairs = []
            for digit_a, digit_b in [(0, 1), (1, 8), (0, 8)]:
                idx_a = (labels == digit_a).nonzero(as_tuple=True)[0]
                idx_b = (labels == digit_b).nonzero(as_tuple=True)[0]
                if len(idx_a) > 0 and len(idx_b) > 0:
                    pairs.append((mu[idx_a[0]], mu[idx_b[0]]))

            if len(pairs) >= 3:
                break

    # Get all latent points for background
    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    plt.figure(figsize=(12, 10))
    plt.scatter(latents[:, 0], latents[:, 1], c='lightgray', s=5, alpha=0.3, label='Data')

    colors = ['red', 'blue', 'green', 'purple', 'orange']

    print("  Computing geodesics...")
    for idx, (z0, z1) in enumerate(pairs[:n_pairs]):
        z0, z1 = z0.to(device), z1.to(device)

        # Euclidean path
        alphas = torch.linspace(0, 1, 30).to(device)
        z_euclidean = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)
        z_euclidean = z_euclidean.cpu().numpy()

        # Geodesic path
        geodesic = compute_geodesic(model, z0, z1, n_waypoints=30, n_steps=100)
        z_geodesic = geodesic.cpu().numpy()

        # Plot
        plt.plot(z_euclidean[:, 0], z_euclidean[:, 1], '--',
                color=colors[idx], alpha=0.5, linewidth=2, label=f'Euclidean {idx+1}')
        plt.plot(z_geodesic[:, 0], z_geodesic[:, 1], '-',
                color=colors[idx], alpha=0.9, linewidth=2.5, label=f'Geodesic {idx+1}')

        # Mark endpoints
        plt.scatter([z0.cpu()[0]], [z0.cpu()[1]], c=colors[idx], s=150,
                   marker='o', edgecolors='black', linewidths=2, zorder=5)
        plt.scatter([z1.cpu()[0]], [z1.cpu()[1]], c=colors[idx], s=150,
                   marker='s', edgecolors='black', linewidths=2, zorder=5)

    plt.xlabel('z₁', fontsize=13)
    plt.ylabel('z₂', fontsize=13)
    plt.title('Geodesic vs Euclidean Paths\n(solid=geodesic, dashed=Euclidean)', fontsize=14)
    plt.legend(loc='best', fontsize=9, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [64]:
def plot_volume_distortion(model, test_loader, save_path='figure_volume_distortion.png'):
    """
    Visualize volume distortion: sqrt(det(M)) shows how volumes are stretched/compressed.
    Important for understanding the geometry of the learned manifold.
    """
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    grid_size = 100
    x_min, x_max = latents[:, 0].min() - 1, latents[:, 0].max() + 1
    y_min, y_max = latents[:, 1].min() - 1, latents[:, 1].max() + 1
    x = np.linspace(x_min, x_max, grid_size)
    y = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x, y)
    Z_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

    print("  Computing volume distortion...")
    vol_factors = []

    z_tensor = torch.FloatTensor(Z_grid).to(device)
    batch_size = 100

    for i in range(0, len(z_tensor), batch_size):
        batch = z_tensor[i:i+batch_size]
        M = model.compute_metric_tensor_simple(batch)
        det = torch.det(M)
        vol = torch.sqrt(torch.abs(det))  # Volume element
        vol_factors.append(vol.cpu().numpy())

    vol_factors = np.concatenate(vol_factors).reshape(grid_size, grid_size)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Linear scale
    im1 = axes[0].contourf(X, Y, vol_factors, levels=30, cmap='plasma')
    axes[0].scatter(latents[:, 0], latents[:, 1], c='white', s=1, alpha=0.5)
    axes[0].set_xlabel('z₁', fontsize=12)
    axes[0].set_ylabel('z₂', fontsize=12)
    axes[0].set_title('Volume Distortion: √det(M)', fontsize=13)
    plt.colorbar(im1, ax=axes[0], label='Volume Factor')

    # Log scale for better contrast
    im2 = axes[1].contourf(X, Y, np.log10(vol_factors + 1e-6), levels=30, cmap='plasma')
    axes[1].scatter(latents[:, 0], latents[:, 1], c='white', s=1, alpha=0.5)
    axes[1].set_xlabel('z₁', fontsize=12)
    axes[1].set_ylabel('z₂', fontsize=12)
    axes[1].set_title('Log Volume Distortion: log₁₀(√det(M))', fontsize=13)
    plt.colorbar(im2, ax=axes[1], label='log₁₀(Volume Factor)')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [65]:
def plot_christoffel_symbols(model, test_loader, save_path='figure_christoffel.png'):
    """
    Visualize Christoffel symbols (connection coefficients).
    Shows how parallel transport works - where geodesics curve.
    """
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    # Sample points
    n_arrows = 15
    x_min, x_max = latents[:, 0].min(), latents[:, 0].max()
    y_min, y_max = latents[:, 1].min(), latents[:, 1].max()

    x_points = np.linspace(x_min, x_max, n_arrows)
    y_points = np.linspace(y_min, y_max, n_arrows)

    plt.figure(figsize=(12, 10))
    plt.scatter(latents[:, 0], latents[:, 1], c='lightgray', s=3, alpha=0.3)

    print("  Computing Christoffel symbols (connection)...")
    eps = 1e-3

    for x in x_points:
        for y in y_points:
            z = torch.FloatTensor([[x, y]]).to(device)

            # Compute metric and its derivatives
            M_0 = model.compute_metric_tensor_simple(z)[0]

            # Derivative w.r.t. z_1
            z_dx = z.clone()
            z_dx[0, 0] += eps
            M_dx = model.compute_metric_tensor_simple(z_dx)[0]
            dM_dx = (M_dx - M_0) / eps

            # Derivative w.r.t. z_2
            z_dy = z.clone()
            z_dy[0, 1] += eps
            M_dy = model.compute_metric_tensor_simple(z_dy)[0]
            dM_dy = (M_dy - M_0) / eps

            # Christoffel symbol Γ^k_ij (simplified: just magnitude)
            gamma_magnitude = (torch.norm(dM_dx) + torch.norm(dM_dy)).item()

            # Arrow showing direction of strongest curvature
            direction = torch.tensor([dM_dx[0, 0].item(), dM_dy[1, 1].item()])
            direction = direction / (torch.norm(direction) + 1e-8)

            # Plot arrow scaled by curvature
            scale = gamma_magnitude * 0.3
            plt.arrow(x, y, direction[0]*scale, direction[1]*scale,
                     head_width=0.1, head_length=0.08, fc='blue', ec='blue',
                     alpha=0.6, linewidth=1.5)

    plt.xlabel('z₁', fontsize=13)
    plt.ylabel('z₂', fontsize=13)
    plt.title('Connection Field (Christoffel Symbols)\nArrows show curvature direction', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")

In [66]:
def plot_parallel_transport(model, z_start, z_end, save_path='figure_parallel_transport.png'):
    """
    Visualize parallel transport along geodesic.
    Shows how vectors are transported while remaining "parallel" in curved space.
    """
    model.eval()

    print("  Computing parallel transport...")

    # Compute geodesic
    geodesic = compute_geodesic(model, z_start, z_end, n_waypoints=20, n_steps=150)
    geo_path = geodesic.cpu().numpy()

    # Initial tangent vector (perpendicular to geodesic direction)
    initial_tangent = (z_end - z_start).cpu().numpy()
    initial_tangent = initial_tangent / (np.linalg.norm(initial_tangent) + 1e-8)
    # Perpendicular vector
    v0 = np.array([-initial_tangent[1], initial_tangent[0]])

    plt.figure(figsize=(12, 10))

    # Plot geodesic
    plt.plot(geo_path[:, 0], geo_path[:, 1], 'b-', linewidth=3, alpha=0.7, label='Geodesic')
    plt.scatter(geo_path[0, 0], geo_path[0, 1], c='green', s=200,
               marker='o', edgecolors='black', linewidths=2, zorder=5, label='Start')
    plt.scatter(geo_path[-1, 0], geo_path[-1, 1], c='red', s=200,
               marker='s', edgecolors='black', linewidths=2, zorder=5, label='End')

    # Parallel transport (simplified: rotate vector based on local metric)
    vectors = [v0]
    for i in range(1, len(geo_path)):
        z_curr = torch.FloatTensor([geo_path[i]]).to(device)
        M = model.compute_metric_tensor_simple(z_curr)[0].cpu().numpy()

        # Simplified parallel transport: project onto tangent space
        v_prev = vectors[-1]
        # Use metric to define "parallel"
        v_new = M @ v_prev
        v_new = v_new / (np.linalg.norm(v_new) + 1e-8)
        vectors.append(v_new)

    # Plot transported vectors
    for i in range(0, len(geo_path), 2):
        scale = 0.3
        plt.arrow(geo_path[i, 0], geo_path[i, 1],
                 vectors[i][0]*scale, vectors[i][1]*scale,
                 head_width=0.08, head_length=0.06,
                 fc='purple', ec='purple', alpha=0.7, linewidth=2)

    plt.xlabel('z₁', fontsize=13)
    plt.ylabel('z₂', fontsize=13)
    plt.title('Parallel Transport Along Geodesic\nPurple vectors remain "parallel" in curved space',
             fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [67]:
def plot_geodesic_deviation(model, test_loader, save_path='figure_geodesic_deviation.png'):
    """
    Measure how much geodesics deviate from straight lines.
    Quantifies the "curvedness" of the space.
    """
    model.eval()

    # Get test point pairs
    with torch.no_grad():
        for data, labels in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)

            pairs = []
            for digit_a, digit_b in [(0, 1), (1, 8), (0, 8)]:
                idx_a = (labels == digit_a).nonzero(as_tuple=True)[0]
                idx_b = (labels == digit_b).nonzero(as_tuple=True)[0]
                if len(idx_a) > 2 and len(idx_b) > 2:
                    for i in range(2):
                        pairs.append((mu[idx_a[i]], mu[idx_b[i]]))

            if len(pairs) >= 6:
                break

    deviations = []
    euclidean_lengths = []
    geodesic_lengths = []

    print("  Computing geodesic deviations...")
    for z0, z1 in pairs[:8]:
        z0, z1 = z0.to(device), z1.to(device)

        # Euclidean path length
        euclidean_len = torch.norm(z1 - z0).item()
        euclidean_lengths.append(euclidean_len)

        # Geodesic
        geodesic = compute_geodesic(model, z0, z1, n_waypoints=30, n_steps=100)

        # Geodesic length
        geo_len = 0
        for i in range(len(geodesic) - 1):
            geo_len += torch.norm(geodesic[i+1] - geodesic[i]).item()
        geodesic_lengths.append(geo_len)

        # Deviation: how far geodesic deviates from straight line
        alphas = torch.linspace(0, 1, len(geodesic)).to(device)
        straight_line = z0.unsqueeze(0) * (1 - alphas).view(-1, 1) + z1.unsqueeze(0) * alphas.view(-1, 1)
        deviation = torch.mean(torch.norm(geodesic - straight_line, dim=1)).item()
        deviations.append(deviation)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Deviation bar chart
    axes[0].bar(range(len(deviations)), deviations, color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Pair Index', fontsize=12)
    axes[0].set_ylabel('Mean Deviation', fontsize=12)
    axes[0].set_title('Geodesic Deviation from Straight Line', fontsize=13)
    axes[0].grid(True, alpha=0.3, axis='y')

    # Length comparison
    x = np.arange(len(euclidean_lengths))
    width = 0.35
    axes[1].bar(x - width/2, euclidean_lengths, width, label='Euclidean', color='orange', alpha=0.7)
    axes[1].bar(x + width/2, geodesic_lengths, width, label='Geodesic', color='purple', alpha=0.7)
    axes[1].set_xlabel('Pair Index', fontsize=12)
    axes[1].set_ylabel('Path Length', fontsize=12)
    axes[1].set_title('Path Length Comparison', fontsize=13)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

    # Ratio
    ratios = np.array(geodesic_lengths) / (np.array(euclidean_lengths) + 1e-8)
    axes[2].bar(range(len(ratios)), ratios, color='crimson', alpha=0.7)
    axes[2].axhline(y=1.0, color='black', linestyle='--', linewidth=2, label='Equal length')
    axes[2].set_xlabel('Pair Index', fontsize=12)
    axes[2].set_ylabel('Geodesic / Euclidean', fontsize=12)
    axes[2].set_title('Length Ratio (>1 means curved space)', fontsize=13)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {save_path}")
    print(f"  Average length ratio: {np.mean(ratios):.3f}")

In [68]:
def plot_anisotropy_map(model, test_loader, save_path='figure_anisotropy.png'):
    """
    Visualize anisotropy: ratio of max/min eigenvalues of metric tensor.
    Shows where space is stretched more in one direction than another.
    """
    model.eval()

    latents = []
    with torch.no_grad():
        for data, label in test_loader:
            data = data.view(-1, 784).to(device)
            mu, _ = model.encode(data)
            latents.append(mu.cpu())
    latents = torch.cat(latents, dim=0).numpy()

    grid_size = 80
    x_min, x_max = latents[:, 0].min() - 1, latents[:, 0].max() + 1
    y_min, y_max = latents[:, 1].min() - 1, latents[:, 1].max() + 1
    x = np.linspace(x_min, x_max, grid_size)
    y = np.linspace(y_min, y_max, grid_size)
    X, Y = np.meshgrid(x, y)
    Z_grid = np.stack([X.ravel(), Y.ravel()], axis=1)

    print("  Computing anisotropy map...")
    anisotropies = []

    z_tensor = torch.FloatTensor(Z_grid).to(device)
    batch_size = 100

    for i in range(0, len(z_tensor), batch_size):
        batch = z_tensor[i:i+batch_size]
        M = model.compute_metric_tensor_simple(batch)

        # Compute eigenvalues
        eigvals = torch.linalg.eigvalsh(M)
        # Anisotropy: ratio of max to min eigenvalue
        aniso = eigvals[:, 1] / (eigvals[:, 0] + 1e-8)
        anisotropies.append(aniso.cpu().numpy())

    anisotropies = np.concatenate(anisotropies).reshape(grid_size, grid_size)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Anisotropy map
    im1 = axes[0].contourf(X, Y, anisotropies, levels=30, cmap='RdYlBu_r')
    axes[0].scatter(latents[:, 0], latents[:, 1], c='black', s=1, alpha=0.3)
    axes[0].set_xlabel('z₁', fontsize=12)
    axes[0].set_ylabel('z₂', fontsize=12)
    axes[0].set_title('Anisotropy Map (λ_max / λ_min)\n1=isotropic, >1=anisotropic', fontsize=13)
    cbar1 = plt.colorbar(im1, ax=axes[0])
    cbar1.set_label('Anisotropy Ratio', fontsize=11)

    # Log scale for better detail
    im2 = axes[1].contourf(X, Y, np.log10(anisotropies), levels=30, cmap='RdYlBu_r')
    axes[1].scatter(latents[:, 0], latents[:, 1], c='black', s=1, alpha=0.3)
    axes[1].set_xlabel('z₁', fontsize=12)
    axes[1].set_ylabel('z₂', fontsize=12)
    axes[1].set_title('Log Anisotropy Map', fontsize=13)
    cbar2 = plt.colorbar(im2, ax=axes[1])
    cbar2.set_label('log₁₀(Anisotropy)', fontsize=11)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

In [69]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_indices = [i for i, (_, label) in enumerate(train_dataset) if label in [0, 1, 8]]
test_indices = [i for i, (_, label) in enumerate(test_dataset) if label in [0, 1, 8]]

train_dataset = Subset(train_dataset, train_indices)
test_dataset = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [70]:
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 18516
Test samples: 3089


In [71]:
model = RiemannianVAE(
    input_dim=784,
    latent_dim=2,
    hidden_dims=[512, 256]
).to(device)

In [72]:
losses = train_vae(
    model,
    train_loader,
    epochs=100,
    lr=1e-3
)

Epoch 10/100, Loss: 114.0155, Beta: 0.180
Epoch 20/100, Loss: 112.3433, Beta: 0.380
Epoch 30/100, Loss: 112.2821, Beta: 0.580
Epoch 40/100, Loss: 112.5452, Beta: 0.780
Epoch 50/100, Loss: 112.8082, Beta: 0.980
Epoch 60/100, Loss: 112.0906, Beta: 1.000
Epoch 70/100, Loss: 111.5001, Beta: 1.000
Epoch 80/100, Loss: 110.9901, Beta: 1.000
Epoch 90/100, Loss: 110.5500, Beta: 1.000
Epoch 100/100, Loss: 109.6298, Beta: 1.000


In [73]:
plot_latent_space(model, test_loader)
plot_variance_landscape(model, test_loader)

 Saved: figure_latent_space.png
  Computing metric landscape...
Saved: figure_variance_analysis.png


In [74]:
with torch.no_grad():
    for data, labels in test_loader:
        data = data.view(-1, 784).to(device)
        mu, _ = model.encode(data)

        idx0 = (labels == 0).nonzero(as_tuple=True)[0]
        idx8 = (labels == 8).nonzero(as_tuple=True)[0]

        if len(idx0) > 0 and len(idx8) > 0:
            z0 = mu[idx0[0]]
            z1 = mu[idx8[0]]
            break

In [75]:
plot_interpolation(model, z0, z1)
plot_geodesic_vs_euclidean_paths(model, test_loader)

   Computing geodesic...
Saved: figure_interpolation.png
  Computing geodesics...
Saved: figure_geodesic_paths.png


In [76]:
plot_random_walk_comparison(model, z0)

  Computing random walks...
  Running Adaptive MCMC...
Saved: figure_random_walk.png
  Final MCMC step size: 0.1126
  Overall acceptance rate: 100.00%


In [77]:
plot_mcmc_diagnostics(model, z0)

  Running MCMC for diagnostics...
 Saved: figure_mcmc_diagnostics.png


In [78]:
plot_metric_ellipses(model, test_loader)
plot_curvature_scalar(model, test_loader)
plot_volume_distortion(model, test_loader)
plot_anisotropy_map(model, test_loader)

  Computing metric ellipses...
Saved: figure_metric_ellipses.png
  Computing scalar curvature...
 Saved: figure_curvature_scalar.png
  Computing volume distortion...
Saved: figure_volume_distortion.png
  Computing anisotropy map...
Saved: figure_anisotropy.png


In [79]:
plot_distance_comparison(model, test_loader)
plot_geodesic_deviation(model, test_loader)

  Computing pairwise distances...
 Saved: figure_distance_comparison.png
  Computing geodesic deviations...
 Saved: figure_geodesic_deviation.png
  Average length ratio: 1.142


In [80]:
plot_christoffel_symbols(model, test_loader)
plot_parallel_transport(model, z0, z1)

  Computing Christoffel symbols (connection)...
 Saved: figure_christoffel.png
  Computing parallel transport...
Saved: figure_parallel_transport.png


In [81]:
plot_rbf_centers(model, train_loader)

Saved: figure_rbf_centers.png
